# Отбор и селекция признаков (ПРАКТИКА)

Задача: 

Обучить модель линейной регрессии на найденных двумя способами трёх важных признаках и сравните полученные результаты.

### Импорт библиотек

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.feature_selection import RFE, SelectKBest, f_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

### Загрузка данных и первичный анализ

In [26]:
ford_price_df = pd.read_excel('Data/data_ford_price.xlsx')
display(ford_price_df.head())
display(ford_price_df.info())

,price,year,condition,cylinders,odometer,title_status,transmission,drive,size,lat,long,weather
0,43900,2016,4,6,43500,clean,automatic,4wd,full-size,36.471500,-82.483400,59.0
1,15490,2009,2,8,98131,clean,automatic,4wd,full-size,40.468826,-74.281734,52.0
2,2495,2002,2,8,201803,clean,automatic,4wd,full-size,42.477134,-82.949564,45.0
3,1300,2000,1,8,170305,rebuilt,automatic,4wd,full-size,40.764373,-82.349503,49.0
4,13865,2010,3,8,166062,clean,automatic,4wd,NaN,49.210949,-123.114720,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7017 entries, 0 to 7016
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         7017 non-null   int64  
 1   year          7017 non-null   int64  
 2   condition     7017 non-null   int64  
 3   cylinders     7017 non-null   int64  
 4   odometer      7017 non-null   int64  
 5   title_status  7017 non-null   object 
 6   transmission  7017 non-null   object 
 7   drive         6626 non-null   object 
 8   size          5453 non-null   object 
 9   lat           7017 non-null   float64
 10  long          7017 non-null   float64
 11  weather       6837 non-null   float64
dtypes: float64(3), int64(5), object(4)
memory usage: 658.0+ KB


None

# Вывод:

Cреди данных можно выделить следующие категориальные признаки:
- 'cylinders'
- 'title_status'
- 'transmission'
- 'drive'
- 'size'

### Проверка на пропуски

In [27]:
display(ford_price_df.isnull().sum())

price              0
year               0
condition          0
cylinders          0
odometer           0
title_status       0
transmission       0
drive            391
size            1564
lat                0
long               0
weather          180
dtype: int64

In [28]:
features = ['size', 'weather', 'drive']
for col in features:
    missing_percent = ford_price_df[col].isna().mean() * 100
    print(f'{col}: {missing_percent:.2f} %')

size: 22.29 %
weather: 2.57 %
drive: 5.57 %


# Вывод:

Среди данных были найдены пропуски в трех признаках:
- drive;
- size;
- weather.

Доля пропусков в признаках не превышаю 30%, несмотря на то, что можно было бы просто удалить пропуски, стоит попробовать заполнить их значениями с помощью ML.

# Заполняем пропуски с помощью ML

In [29]:
def encode_cat_features(columns_to_change, X_train, X_test):
    # Проведём кодирование OneHot-методом категориальных переменных.
    one_hot_encoder = OneHotEncoder(handle_unknown='ignore')
    # Обучаем энкодер и сразу применяем преобразование к выборке. Результаты переводим в массивы
    X_train_onehot = one_hot_encoder.fit_transform(X_train[columns_to_change]).toarray()
    X_test_onehot = one_hot_encoder.transform(X_test[columns_to_change]).toarray()
    
    # Для удобства сохраним полученные названия новых колонок в отдельную переменную
    columns = one_hot_encoder.get_feature_names_out(columns_to_change)
    
    # Теперь у нас есть массив закодированных признаков и наша изначальная таблица. 
    # Чтобы соединить эти данные, переведём массив в формат DataFrame.
    X_train_onehot_df = pd.DataFrame(X_train_onehot, columns=columns)
    X_test_onehot_df = pd.DataFrame(X_test_onehot, columns=columns)

    X_train = X_train.reset_index().drop(['index'], axis = 1)
    X_test = X_test.reset_index().drop(['index'], axis = 1)

    # Объединяем таблицы и удаляем старые категориальные признаки
    X_train_new = pd.concat([X_train, X_train_onehot_df], axis=1)
    X_test_new = pd.concat([X_test, X_test_onehot_df], axis=1)
    
    X_train_new = X_train_new.drop(columns=columns_to_change)
    X_test_new = X_test_new.drop(columns=columns_to_change)

    return X_train_new, X_test_new

In [30]:
def fill_numeric_feature(X, feature, categorical_features):
    # Выделяем строки с пропущенными значениями в целевом признаке
    test_data = X[X[feature].isnull()]
    
    # Если пропусков нет — выходим без изменений
    if test_data.empty:
        print(f"[OK] Признак '{feature}' не содержит пропусков.")
        return X
    
    # Выделяем обучающую выборку (строки, где пропусков нет)
    train_data = X[X[feature].notnull()]

    # Целевая переменная — это сам признак, который нужно заполнить
    y_train = train_data[feature]
    
    # Удаляем целевой признак из обучающей и тестовой выборок (он не должен быть среди признаков)
    X_train = train_data.drop(columns=[feature])
    X_test = test_data.drop(columns=[feature])

    # Фильтруем список категориальных признаков: используем только те, что действительно есть в X_train
    categorical_cols = [col for col in categorical_features if col in X_train.columns]

    # Кодируем категориальные признаки one-hot методом
    X_train_enc, X_test_enc = encode_cat_features(categorical_cols, X_train, X_test)

    # Обучаем модель линейной регрессии (для числовых значений)
    model = LinearRegression()
    model.fit(X_train_enc, y_train)

    # Предсказываем пропущенные значения
    y_pred = model.predict(X_test_enc)

    # Вставляем предсказанные значения обратно в исходный датафрейм X
    for i, idx in enumerate(test_data.index):
        X.loc[idx, feature] = y_pred[i]

    print(f"[OK] Заполнено {len(test_data)} пропусков в числовом признаке '{feature}'")
    
    return X  # Возвращаем датафрейм с заполненными значениями

In [31]:
def fill_categorical_feature(X, feature, categorical_features):
    # Выделяем строки, где целевой признак содержит пропуски
    test_data = X[X[feature].isnull()]

    # Если таких строк нет — завершаем выполнение функции
    if test_data.empty:
        print(f"[OK] Признак '{feature}' не содержит пропусков.")
        return X

    # Формируем обучающую выборку: строки, где значения есть
    train_data = X[X[feature].notnull()]

    # Целевой признак — это сам признак, который мы хотим восстановить
    y_train = train_data[feature]

    # Удаляем целевой признак из обучающей и тестовой выборок — он не должен быть в признаках
    X_train = train_data.drop(columns=[feature])
    X_test = test_data.drop(columns=[feature])

    # Определяем список категориальных признаков, присутствующих в обучающей выборке
    categorical_cols = [col for col in categorical_features if col in X_train.columns]

    # Применяем one-hot-кодирование к категориальным признакам
    X_train_enc, X_test_enc = encode_cat_features(categorical_cols, X_train, X_test)

    # Создаём и обучаем модель классификации
    model = RandomForestClassifier()
    model.fit(X_train_enc, y_train)

    # Предсказываем пропущенные значения
    y_pred = model.predict(X_test_enc)

    # Вставляем предсказанные значения обратно в исходный датафрейм
    for i, idx in enumerate(test_data.index):
        X.loc[idx, feature] = y_pred[i]

    # Выводим отчёт о количестве заполненных строк
    print(f"[OK] Заполнено {len(test_data)} пропусков в категориальном признаке '{feature}'")

    # Возвращаем датафрейм с заполненными значениями
    return X

In [32]:
X = ford_price_df.drop(columns='price')
y = ford_price_df['price']

In [33]:
categorical_features_all = ['cylinders', 'title_status', 'transmission', 'drive', 'size']

# Заполняем числовые
X = fill_numeric_feature(X, 'weather', categorical_features_all)

# Заполняем категориальный
X = fill_categorical_feature(X, 'size', categorical_features_all)
X = fill_categorical_feature(X, 'drive', categorical_features_all)

# Проверим результат
display(X.isnull().sum())

[OK] Заполнено 180 пропусков в числовом признаке 'weather'
[OK] Заполнено 1564 пропусков в категориальном признаке 'size'
[OK] Заполнено 391 пропусков в категориальном признаке 'drive'


year            0
condition       0
cylinders       0
odometer        0
title_status    0
transmission    0
drive           0
size            0
lat             0
long            0
weather         0
dtype: int64

# Вывод:

Теперь все пропущенные значения заполнены с помощью двух функций с моделями LinearRegression, RandomForestClassifier.

# Кодируем категориальные признаки

Разделим выборку на тестовую и обучающую

In [34]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [35]:
categorical_features = ['title_status', 'transmission', 'drive', 'size']
X_train_new, X_test_new = encode_cat_features(categorical_features, X_train, X_test)

# Выбросы

Перед тем, как будем проводить маштабирование, проверим данные на выбросы и постараемся от них избавиться.

In [36]:
# ищем выбросы в обучающей выборке
iso = IsolationForest(contamination=0.1)
iso.fit(X_train_new.values)
y_predicted = iso.predict(X_train_new.values)

# выберем все строки, которые не являются выбросами
mask = y_predicted != -1
X_train_new, y_train = X_train_new[mask], y_train[mask]

print(X_train_new.shape, y_train.shape)

(4421, 22) (4421,)


# Маштабирование

Поскольку мы избавились примерно от 10% выбросов, далее проведем стандартизацию с помощью StandardScaler.

In [37]:
scaler = StandardScaler()

# Обучаем скейлер на обучающих данных и применяем к ним
X_train_scaled = scaler.fit_transform(X_train_new)

# Применяем те же параметры (без повторного обучения) к тестовой выборке
X_test_scaled = scaler.transform(X_test_new)

print("Среднее по обучающим признакам:", np.round(X_train_scaled.mean(axis=0), 2))
print("Ст. отклонение по обучающим признакам:", np.round(X_train_scaled.std(axis=0), 2))

Среднее по обучающим признакам: [-0. -0.  0.  0. -0. -0.  0.  0. -0. -0. -0. -0.  0. -0.  0. -0.  0. -0.
 -0. -0. -0.  0.]
Ст. отклонение по обучающим признакам: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0.]


# Превратим X_train_scaled и X_test_scaled обратно в DataFrame с именами признаков

In [38]:
X_train = pd.DataFrame(X_train_scaled, columns=X_train_new.columns, index=X_train_new.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test_new.columns, index=X_test_new.index)

# RFE (отбор признаков для модели)

In [39]:
# Импортируем модель линейной регрессии, которая будет использоваться в качестве оценщика (estimator)
estimator = LinearRegression()

# Инициализируем рекурсивное исключение признаков (RFE):
# - estimator: модель, по которой оценивается важность признаков
# - n_features_to_select=3: хотим оставить только 3 признака
# - step=1: удаляем по одному признаку за итерацию
selector = RFE(estimator, n_features_to_select=3, step=1)

# Обучаем RFE на обучающих данных
# Во время обучения RFE будет поочерёдно исключать наименее значимые признаки
selector = selector.fit(X_train, y_train)

# Получаем имена выбранных признаков (если X_train — DataFrame с именами колонок)
selector.get_feature_names_out()

array(['year', 'odometer', 'drive_4wd'], dtype=object)

In [40]:
X_train_rfe = X_train[selector.get_feature_names_out()]
X_test_rfe = X_test[selector.get_feature_names_out()]

model = LinearRegression()
model.fit(X_train_rfe, y_train)

y_pred = model.predict(X_test_rfe)

mae = mean_absolute_error(y_test, y_pred)
r_2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"R_2: {r_2:.3f}")

MAE: 5104.76
R_2: 0.443


# SelectKBest (отбор признаков для модели)

In [41]:
# Импортируем селектор признаков, использующий тест F для регрессии
selector = SelectKBest(f_regression, k=3)  # Выбираем 3 признака с наибольшей F-статистикой

# Обучаем селектор на тренировочных данных
selector.fit(X_train, y_train)  # X_train — признаки, y_train — целевая переменная

# Вывод наиболее важных признаков 
selector.get_feature_names_out()

scores = pd.Series(selector.scores_, index=X_train.columns)
print(scores.sort_values(ascending=False))

year                      5865.257792
odometer                  1681.896406
condition                 1009.048433
cylinders                  700.583701
drive_4wd                  487.666943
drive_rwd                  470.544908
size_mid-size              238.921366
size_full-size             234.588008
lat                        122.290405
weather                     92.007922
long                        70.467193
transmission_manual         11.521345
drive_fwd                    8.775829
transmission_automatic       6.298251
title_status_missing         1.454204
title_status_salvage         1.110438
title_status_rebuilt         0.925429
size_compact                 0.752638
title_status_clean           0.036912
title_status_lien            0.009448
transmission_other           0.001138
size_sub-compact             0.000000
dtype: float64


In [42]:
X_train_kbest = X_train[selector.get_feature_names_out()]
X_test_kbest = X_test[selector.get_feature_names_out()]

model = LinearRegression()
model.fit(X_train_kbest, y_train)

y_pred = model.predict(X_test_kbest)

mae = mean_absolute_error(y_test, y_pred)
r_2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"R_2: {r_2:.3f}")

MAE: 5127.55
R_2: 0.453


### Сравнение результатов моделей

Модель, обученная на признаках, выбранных с помощью **SelectKBest**, показала следующие результаты на тестовой выборке:
- **MAE**: 5127.55
- **R²**: 0.453

Для сравнения, модель с признаками, выбранными с помощью **RFE**, показала:
- **MAE**: 5104.76
- **R²**: 0.443

### Вывод

Обе модели показали **сопоставимое качество**, однако:
- **SelectKBest** дал чуть **лучший R²**, что говорит о большей объясняющей способности признаков.
- **RFE** дал немного **меньший MAE**, то есть модель ошибается чуть меньше в абсолютных значениях.

В условиях этой задачи **разница несущественна**, но **SelectKBest** показывает более стабильный результат при использовании линейной регрессии.